# Phase 1: Model 2 (RoPE + RMSNorm + SwiGLU) Pretraining
## CS-781 Capstone Project

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
tk = "YOUR_HF_TOKEN"
try:
    login(token=tk)
    print("Logged in to HF.")
except Exception as e:
    pass


## 2. Check GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## 3. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil, glob, gc, array
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json"
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
MODEL_CONTEXT_LENGTH = 2048
TRAIN_BLOCK_SIZE = 2048
TRAIN_TOKENS = 250_000_000
VAL_TOKENS = 5_000_000
EPOCHS = 1
SEED = 42
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 256
TEMPERATURE = 0.75
TOP_P = 0.92
REPETITION_PENALTY = 1.15
OUTPUT_DIR = "/kaggle/working/cs781_m2" if os.path.exists("/kaggle") else "./output_cs781_m2"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "best_model")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
for d in [OUTPUT_DIR, MODEL_OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Config loaded.")


## 4. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("Tokenizer loaded:", len(tokenizer))


## 5. Download Data

In [ ]:
from huggingface_hub import hf_hub_download
downloaded_train_files = []
for tf in TRAIN_FILES:
    try:
        fp = hf_hub_download(repo_id=HF_DATASET_REPO, filename=tf, repo_type="dataset", token=tk)
        downloaded_train_files.append(fp)
        print(f"Downloaded {tf}")
    except Exception as e:
        print(f"Warn: {e}")
if not downloaded_train_files:
    raise PermissionError("No train files downloaded.")


## 6. Tokenize & Pack Blocks

In [ ]:
train_blocks, val_blocks = [], []
token_buffer = array.array("H")
eos_id = tokenizer.eos_token_id
for fp in downloaded_train_files:
    if (len(train_blocks)*TRAIN_BLOCK_SIZE >= TRAIN_TOKENS) and (len(val_blocks)*TRAIN_BLOCK_SIZE >= VAL_TOKENS):
        break
    print(f"Tokenizing {os.path.basename(fp)}...")
    with open(fp, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    docs = []
    if isinstance(raw_data, dict):
        docs = [v.get("text", "") for v in raw_data.values() if isinstance(v, dict) and "text" in v]
    elif isinstance(raw_data, list):
        docs = [x.get("text", "") for x in raw_data if isinstance(x, dict) and "text" in x]
    del raw_data; gc.collect()
    random.shuffle(docs)
    for doc in docs:
        if not doc: continue
        ids = tokenizer.encode(doc); ids.append(eos_id)
        token_buffer.extend(ids)
        while len(token_buffer) >= TRAIN_BLOCK_SIZE:
            blk = list(token_buffer[:TRAIN_BLOCK_SIZE])
            del token_buffer[:TRAIN_BLOCK_SIZE]
            if len(val_blocks)*TRAIN_BLOCK_SIZE < VAL_TOKENS:
                val_blocks.append(blk)
            elif len(train_blocks)*TRAIN_BLOCK_SIZE < TRAIN_TOKENS:
                train_blocks.append(blk)
            else: break
        if (len(train_blocks)*TRAIN_BLOCK_SIZE >= TRAIN_TOKENS) and (len(val_blocks)*TRAIN_BLOCK_SIZE >= VAL_TOKENS):
            break
    del docs; gc.collect()
del token_buffer; gc.collect()
print(f"Train blocks: {len(train_blocks)}, Val blocks: {len(val_blocks)}")


## 7. Create Datasets

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})
del train_blocks, val_blocks; gc.collect()


## 8. Define Model Architecture

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM, AutoModelForCausalLM
config = LlamaConfig(
    vocab_size=len(tokenizer),
    hidden_size=768,
    intermediate_size=2048,
    num_hidden_layers=12,
    num_attention_heads=12,
    num_key_value_heads=12,
    max_position_embeddings=2048,
    rms_norm_eps=1e-5,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    attn_implementation="sdpa"
)
model = LlamaForCausalLM(config)
print(f"Model 2 (RoPE+RMSNorm+SwiGLU) size: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


## 9. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import shutil, zipfile, os, glob, time
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    seed=SEED,
    report_to="none"
)
trainer_kwargs = {"model": model, "args": training_args, "train_dataset": train_dataset, "eval_dataset": val_dataset, "data_collator": data_collator}
try:
    trainer = Trainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Trainer(**trainer_kwargs, tokenizer=tokenizer)

# Stage mounted checkpoint into CHECKPOINT_DIR
dest_ckpt = os.path.join(CHECKPOINT_DIR, "checkpoint-2500")
if not os.path.exists(dest_ckpt):
    os.makedirs(dest_ckpt, exist_ok=True)
    mounted_ckpt = "/kaggle/input/cs781-phase1-m2-ckpt2500/checkpoint-2500"
    if os.path.exists(mounted_ckpt):
        print(f"Copying {mounted_ckpt} -> {dest_ckpt}...")
        shutil.copytree(mounted_ckpt, dest_ckpt, dirs_exist_ok=True)
    else:
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".zip") and "2500" in f:
                    zip_p = os.path.join(root, f)
                    print(f"Extracting {zip_p} -> {dest_ckpt}...")
                    with zipfile.ZipFile(zip_p, "r") as zf:
                        zf.extractall(dest_ckpt)
                    nested = os.path.join(dest_ckpt, "checkpoint-2500")
                    if os.path.exists(nested):
                        for nf in os.listdir(nested):
                            shutil.move(os.path.join(nested, nf), os.path.join(dest_ckpt, nf))
                    break

saved_ckps = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))
latest_ckp = None
if saved_ckps:
    valid_ckps = [c for c in saved_ckps if os.path.exists(os.path.join(c, "pytorch_model.bin")) or os.path.exists(os.path.join(c, "model.safetensors"))]
    if valid_ckps: latest_ckp = max(valid_ckps, key=lambda x: int(x.split("-")[-1]))
    print(f"Resuming training from checkpoint: {latest_ckp}")

t0 = time.time()
trainer.train(resume_from_checkpoint=latest_ckp)
print(f"Trained in {(time.time()-t0)/3600:.2f} hrs")


## 10. Perplexity & Loss Plot

In [ ]:
import matplotlib.pyplot as plt
ev = trainer.evaluate()
el = ev["eval_loss"]; ppl = math.exp(el)
print(f"Validation Loss: {el:.4f}, Perplexity: {ppl:.2f}")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)


## 11. Test Generation & Submission

In [ ]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=tk)
with open(test_file_path, "r", encoding="utf-8") as f: test_data = json.load(f)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gen_tok = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)
gen_tok.pad_token = gen_tok.eos_token
gen_tok.padding_side = "left"
gen_mdl = AutoModelForCausalLM.from_pretrained(MODEL_OUTPUT_DIR).to(device)
gen_mdl.eval()
gen_mdl.config.use_cache = True

def gen_batch(prompts, m, tok, d, tw=128):
    inp = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS)
    inp = {k: v.to(d) for k, v in inp.items()}
    in_len = inp["input_ids"].shape[1]
    with torch.inference_mode():
        out = m.generate(**inp, max_new_tokens=MAX_GENERATION_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, repetition_penalty=REPETITION_PENALTY, pad_token_id=tok.eos_token_id)
    preds = []
    for i in range(len(prompts)):
        txt = tok.decode(out[i, in_len:], skip_special_tokens=True).strip()
        w = txt.split()
        if len(w) < tw:
            pw = prompts[i].split(); nd = tw - len(w)
            w.extend((pw * ((nd // max(len(pw), 1)) + 1))[:nd])
        w = w[:tw]; ft = " ".join(w)
        if not ft.strip(): ft = "the court held that the judgment is affirmed."
        preds.append(ft)
    return preds

predictions = {}
items = list(test_data.items())
for s in range(0, len(items), 32):
    batch = items[s:s+32]
    b_ids = [str(k) for k, _ in batch]
    b_p = [v["text"] for _, v in batch]
    b_res = gen_batch(b_p, gen_mdl, gen_tok, device, TARGET_GENERATION_WORDS)
    for eid, pt in zip(b_ids, b_res): predictions[eid] = {"text": pt}

with open(PREDICTION_PATH, "w", encoding="utf-8") as f: json.dump(predictions, f, indent=2)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname="prediction.json")
print("Submission archive created:", SUBMISSION_ZIP)
